In [1]:
#include <iostream>
#include <fstream>
#include <vector>
#include <TGraph.h>
#include <TCanvas.h>
#include <string>
#include <cmath>
#include <array>
#include <algorithm>

The U.S. Standard Atmosphere, 1976 model computes atmospheric properties (pressure, density, temperature) based on altitude ($h$), defined from sea level up to 1000 km. It is primarily determined by a defined temperature profile ($T$ vs $h$), using the hydrostatic equation and ideal gas law to calculate pressure and density across different layers. [1, 2, 3, 4, 5]  
Key Sea Level Constants (0 km) 

• Pressure (): $101325 \text{ Pa}$ ($1.01325 \times 10^5 \text{ N/m}^2$). 
• Temperature (): $288.15 \text{ K}$ ($15 \text{ °C}$ or $59 \text{ °F}$). 
• Density (): $1.2250 \text{ kg/m}^3$. 
• Gravitational Acceleration (): $9.80665 \text{ m/s}^2$. [2, 6]  

Core Equations (Troposphere to Stratosphere) 
The model splits the atmosphere into layers with specific temperature gradients ($L_m = \frac{dT_m}{dh}$). [1, 4]  
1. Temperature () at Geopotential Altitude (): $T_m = T_b + L_m(h - h_b)$ 
Where  is the reference temperature at layer bottom, and  is the lapse rate. 
2. Pressure () Calculation: 

• If lapse rate : $P = P_b \left[\frac{T_b}{T_b + L_m(h-h_b)}\right]^{\frac{g_0 M}{R^* L_m}}$ 

• If lapse rate  (Isothermal): $P = P_b \exp\left[\frac{-g_0 M (h-h_b)}{R^* T_b}\right]$ 

3. Density () Calculation: $\rho = \frac{P M}{R^* T_m}$ 

• : Molecular weight of air ($28.9644 \text{ kg/kmol}$). 
• : Universal gas constant ($8314.32 \text{ J/(kmol} \cdot \text{K)}$). 
• : Geopotential altitude at the base of the layer. [4, 6, 8, 9, 10]  

Key Layers (0 to ~86 km) [2]  
The 1976 model provides specific gradients ($L_m$) for calculating $T$ and $P$ within these layers: 

• Troposphere (): Temperature decreases by $6.5 \text{ K/km}$. 
• Lower Stratosphere (): Temperature is constant ($216.65 \text{ K}$). 
• Upper Stratosphere (): Various gradients define increasing and then decreasing temperature up to the mesopause. [2, 4, 11, 12, 13]  

Note: The 1976 Standard Atmosphere is generally identical to ICAO Standard (1964) up to  and ISO Standard (1973) to . [1]  

In [2]:
/**
 * @brief Computes ambient atmospheric pressure based on the U.S. Standard Atmosphere (1976).
 * @param geometric_altitude_m The altitude of the rocket/craft in meters above sea level.
 * @return The ambient pressure in Pascals (Pa). Returns 0.0 if out of atmospheric bounds (>86km).
 */
double get_ambient_pressure(double geometric_altitude_m, double EARTH_RADIUS = 6378100.0) {
    // 1. Convert geometric altitude to geopotential altitude (accounting for gravity drop)
    double h = (EARTH_RADIUS * geometric_altitude_m) / (EARTH_RADIUS + geometric_altitude_m);

    if (h < 0.0) return 101325.0; // Cap at Sea Level pressure if underwater/below datum
    if (h > 86000.0) return 0.0;  // Beyond the 1976 model boundary (effectively vacuum)

    // 2. Define structural data layers for the U.S. Standard Atmosphere 1976
    struct AtmosphericLayer {
        double base_h;  // Geopotential base altitude (m)
        double base_P;  // Base pressure of layer (Pa)
        double base_T;  // Base temperature of layer (K)
        double lapse_L; // Temperature lapse rate (K/m)
    };

    // Pre-calculated exact historical base pressures to prevent structural drift error
    static const std::array<AtmosphericLayer, 7> layers = {{
        {0.0,     101325.0,   288.15, -0.0065}, // Troposphere
        {11000.0, 22632.1,    216.65,  0.0000}, // Tropopause / Lower Stratosphere
        {20000.0, 5474.89,    216.65,  0.0010}, // Stratosphere
        {32000.0, 868.019,    228.65,  0.0028}, // Stratosphere Upper
        {47000.0, 110.906,    270.65,  0.0000}, // Stratopause
        {51000.0, 66.9389,    270.65, -0.0028}, // Mesosphere
        {71000.0, 3.95642,    214.65, -0.0020}  // Mesosphere Upper
    }};

    // 3. Find the current operational layer using binary search (std::upper_bound)
    auto it = std::upper_bound(layers.begin(), layers.end(), h, 
        [](double val, const AtmosphericLayer& layer) { return val < layer.base_h; });
    
    // Step back to grab the active layer lower bound
    const auto& layer = *(std::prev(it));

    // 4. Physical Constants
    constexpr double g0 = 9.80665;      // Standard gravity (m/s^2)
    constexpr double M  = 0.0289644;    // Molar mass of Earth's air (kg/mol)
    constexpr double R  = 8.31432;      // Universal gas constant (J/mol*K)

    // 5. Compute Pressure based on local thermodynamic profile
    double delta_h = h - layer.base_h;

    if (std::abs(layer.lapse_L) < 1e-9) {
        // Isothermal Layer (Lapse rate is 0) -> Governed purely by the Boltzmann/Hydrostatic variant
        return layer.base_P * std::exp((-g0 * M * delta_h) / (R * layer.base_T));
    } else {
        // Lapsing Layer (Temperature changes linearly) -> Governed by Power-Law variant
        double T_local = layer.base_T + layer.lapse_L * delta_h;
        double exponent = (-g0 * M) / (R * layer.lapse_L);
        return layer.base_P * std::pow(T_local / layer.base_T, exponent);
    }
}


## ISRO LVM3 rocket stages and mass update function

In [3]:
/**
 * @brief Updates rocket mass and returns active engine parameters based on altitude.
 * @param altitude_m Current height of the vehicle above sea level.
 * @param current_mass Reference to the vehicle mass (will be modified instantly when dropping empty stages).
 * @param mass_flow_rate Output parameter: Combined mass flow rate of active engines (kg/s).
 * @param exit_velocity Output parameter: Effective exhaust velocity (m/s).
 * @param exit_pa Output parameter: Nozzle exit plane pressure (Pa).
 * @param nozzle_exit_area Output parameter: Combined structural nozzle exit area (m²).
 */
void get_lvm3_stage_properties(double altitude_m, double& current_mass,
                               double& mass_flow_rate, double& exit_velocity,
                               double& exit_pa, double& nozzle_exit_area) 
{
    // Static flags to ensure stage structural masses are only dropped ONCE at transition points
    static bool s200_jettisoned = false;
    static bool l110_jettisoned = false;

    // Phase 1: S200 Solid Strap-ons Active (0 to 43,000 meters)
    if (altitude_m < 43000.0) {
        mass_flow_rate   = 5460.0;  // Twin S200 flow rate
        exit_velocity    = 2690.0;
        exit_pa          = 60000.0; // Optimized near sea-level
        nozzle_exit_area = 11.2;    // Massive combined solid nozzle exit area
    }
    // Phase 2: S200 + L110 Air-Lit Core Overlap (43,000 to 60,000 meters)
    else if (altitude_m >= 43000.0 && altitude_m < 60000.0) {
        mass_flow_rate   = 6031.4;  // Combined Solid (5460.0) + Liquid (571.4)
        exit_velocity    = 2710.0;  // Weighted average thrust velocity
        exit_pa          = 52000.0; // Mixed pressure profile
        nozzle_exit_area = 12.862;  // Combined solid and liquid exit areas
    }
    // Phase 3: L110 Liquid Vikas Core Active (60,000 to 175,000 meters)
    else if (altitude_m >= 60000.0 && altitude_m < 175000.0) {
        // Handle physical detachment event at the boundary threshold
        if (!s200_jettisoned) {
            current_mass -= 62000.0; // Instantly drop dry mass of twin empty solid casings (~31t each)
            s200_jettisoned = true;
        }
        mass_flow_rate   = 571.4;   // Twin Vikas liquid core engines running only
        exit_velocity    = 2873.0;  // High vacuum velocity
        exit_pa          = 52000.0; // Fixed nozzle structural exit pressure
        nozzle_exit_area = 1.662;   // Combined twin Vikas exit area
    }
    // Phase 4: C25 Cryogenic Upper Stage Active (>175,000 meters)
    else {
        // Handle physical detachment event at the boundary threshold
        if (!l110_jettisoned) {
            current_mass -= 105000.0; // Instantly drop L110 structural dry mass + payload fairing
            l110_jettisoned = true;
        }
        mass_flow_rate   = 44.5;    // CE-20 Cryogenic Engine consumption flow
        exit_velocity    = 4345.0;  // Highly efficient hydrogen-oxygen vacuum velocity
        exit_pa          = 5000.0;  // Engineered vacuum optimization expansion
        nozzle_exit_area = 2.45;    // Large single cryogenic nozzle expansion area
    }
}


In [3]:
auto x = 0.53

(double) 0.53000000


In [5]:
auto y = 0.72

(double) 0.72000000


In [7]:
for(int i = 0; i < 9;i ++) {
    x *= 1.001;
}
x

(double) 0.53478912


In [9]:
printf("result is %f \n", x + y)

result is 1.254789 
(int) 20


In [23]:
acos(sqrt(3)/2) // pi/6 rad

(double) 0.52359878


In [11]:
sqrt(x)

(double) 0.73129278


The TMath namespace in the [ROOT data analysis framework](https://root.cern/manual/math/) provides a comprehensive set of free functions for numerical, statistical, and geometrical calculations. While many functions are optimized wrappers for the standard C++ <cmath> library, others offer specialized algorithms or increased precision. [1, 2] 
## Core Categories of TMath Functions## 1. Elementary Math and Trigonometry
These cover basic arithmetic operations and trigonometric functions. They are often used within TF1 (1D function) or [TFormula](https://root.cern/doc/v628/classTFormula.html) definitions. [1, 3, 4] 

* Elementary: TMath::Sqrt(x), TMath::Exp(x), TMath::Log(x), TMath::Power(x, y), TMath::Abs(x).
* Trigonometric: TMath::Sin(x), TMath::Cos(x), TMath::Tan(x), and their inverses (ASin, ACos, ATan).
* Hyperbolic: TMath::SinH(x), TMath::CosH(x), TMath::TanH(x). [5, 6, 7, 8, 9] 

## 2. Statistical Distributions
TMath includes many [predefined probability density functions (PDFs)](https://root.cern.ch/doc/master/namespaceTMath.html) and cumulative distributions. [1, 10] 

* Common Distributions: TMath::Gaus(x, mean, sigma), TMath::Poisson(x, par), TMath::Landau(x, mpv, sigma), TMath::Binomial(n, k).
* Statistical Tests: TMath::Prob(chi2, ndf) and TMath::KolmogorovProb(z).
* Quantiles: TMath::NormQuantile(p), TMath::ChisquareQuantile(p, ndf). [5, 7, 11, 12, 13] 

## 3. Algorithms for Arrays
These functions are designed to operate on C-style arrays or C++ iterators. [1, 2, 14] 

* Statistics: TMath::Mean(), TMath::Median(), TMath::RMS(), TMath::StdDev().
* Searching & Sorting: TMath::Sort(), TMath::BinarySearch(), TMath::LocMin(), TMath::LocMax(). [1, 2, 7, 15] 

## 4. Special Functions
These are advanced mathematical routines often used in high-energy physics. [1, 2, 10] 

* Bessel Functions: TMath::BesselJ0(x), TMath::BesselI(n, x), TMath::BesselK(n, x).
* Error & Gamma Functions: TMath::Erf(x), TMath::Erfc(x), TMath::Gamma(z), TMath::LnGamma(z).
* ROOT-Specific Specials: TMath::DiLog(x) and TMath::StruveH0(x) (not typically found in standard C++ libraries). [1, 7, 11, 15, 16] 

## Numerical Constants
TMath also provides physical and mathematical constants as inline functions rather than macros: [2, 16] 

* TMath::Pi(), TMath::E(), TMath::C() (Speed of Light).
* TMath::K() (Boltzmann constant), TMath::G() (Gravitational constant), TMath::H() (Planck's constant). [5, 7, 11] 

## Usage Notes

* Modern Preference: For many special functions and distributions, ROOT now encourages using the newer ROOT::Math namespace (part of MathCore) for better performance and alignment with modern standards.
* Headers: To use these functions in a standalone C++ script, you must include the [TMath header](https://root.cern/doc/v612/TMath_8h.html):

#include "TMath.h"double val = TMath::Gaus(x, 0, 1);

[1, 2, 12, 16] 


[1] [https://root.cern](https://root.cern/manual/math/)
[2] [https://root.cern.ch](https://root.cern.ch/root/htmldoc/guides/users-guide/MathLibraries.html)
[3] [https://root.cern.ch](https://root.cern.ch/doc/master/classTF1.html)
[4] [https://root.cern](https://root.cern/doc/v628/classTFormula.html)
[5] [https://root.cern](https://root.cern/root/html602/TMath.html)
[6] [https://www.fuw.edu.pl](https://www.fuw.edu.pl/~kpias/nkfj/root_intro.pdf)
[7] [https://github.com](https://github.com/cxx-hep/root-cern/blob/master/math/mathcore/inc/TMath.h)
[8] [https://root.cern](https://root.cern/doc/v628/namespaceTMath.html)
[9] [https://www.scaler.com](https://www.scaler.com/topics/cmath-cpp/)
[10] [https://root.cern](https://root.cern/doc/v622/namespaceTMath.html)
[11] [https://root.cern.ch](https://root.cern.ch/doc/master/namespaceTMath.html)
[12] [https://root-forum.cern.ch](https://root-forum.cern.ch/t/using-predefined-functions-from-tmath/1842)
[13] [https://root.cern](https://root.cern/doc/v628/TMath_8h.html)
[14] [https://root-forum.cern.ch](https://root-forum.cern.ch/t/how-to-use-tmath-mean/50824)
[15] [https://root.cern.ch](https://root.cern.ch/root/html522/TMath.html)
[16] [https://root.cern](https://root.cern/root/html602/math/mathcore/TMath.html)


In [15]:
double mu = 0.0; double sigma = 1.0;

In [19]:
auto random_from_gauss_distr= gRandom->Gaus(mu, sigma)

(double) 0.99893272


In [21]:
TMath::Cos(3)

(double) 1.7320508


In [4]:
%jsroot on

Info in <TCanvas::MakeDefCanvas>:  created default TCanvas with name c1


In [9]:
TCanvas *c1 = new TCanvas("c1", "Canvas", 800, 600);
auto h2 = new TH1F("h2", "Gaussian Dist;X-axis;Events", 100, -5, 5);
h2->FillRandom("gaus", 1000000);
h2->Draw();
c1->Draw();

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Warning in <TROOT::Append>: Replacing existing TH1: h2 (Potential memory leak).


In [6]:
auto h1 = new TH1I("histname", "Histogram Title", 10, 0., 100.);
h1->SetBinContent(5, 40);
h1->Draw();

In [7]:
c1->Draw();

In [8]:
h1->SetBinContent(1, 20);
h1->SetBinContent(2, 70);
h1->SetBinContent(3, 80);
h1->SetBinContent(4, 52);
h1->SetBinContent(6, 93);
h1->SetBinContent(1, 67);
h1->Draw();
c1->Draw();

In [11]:
auto c2 = new TCanvas("c1", "Heat Equation GIF", 800, 600);
auto heat = new TF1("heat", "1.0/sqrt(4*pi*[1]*[0]) * exp(-(x*x)/(4*[1]*[0]))", -10, 10);

double alpha = 0.5;
heat->SetParameter(1, alpha);
heat->SetLineColor(kBlue);
heat->SetMinimum(0);
heat->SetMaximum(1);

int nFrames = 30;
for (int i = 1; i <= nFrames; ++i) {
    double t = i * 0.2;
    heat->SetParameter(0, t);
    heat->SetTitle(Form("Time t = %.1f;x;u(x,t)", t));
    
    heat->Draw();
    c2->Modified();
    c2->Update();

    if (i == 1) {
        // Start the GIF
        c2->Print("heat_evolution.gif");
    } else if (i == nFrames) {
        // Add the last frame with a '+' 
        // Some systems prefer a small delay here
        c2->Print("heat_evolution.gif+");
    } else {
        // Append frame with '+'
        c2->Print("heat_evolution.gif+");
    }
}

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: gif file heat_evolution.gif has been created


In [25]:
%%html
<img src="heat_evolution.gif" />

In [22]:
auto c3 = new TCanvas("c3", "3D Heat Evolution", 800, 600);

// [0] = time 't', [1] = alpha
auto heat3D = new TF2("heat3D", "(1.0/(4*pi*[1]*[0])) * exp(-(x*x + y*y)/(4*[1]*[0]))", -5, 5, -5, 5);

double alpha = 0.5;
heat3D->SetParameter(1, alpha);
heat3D->SetTitle("3D Heat Diffusion;X;Y;Temperature");

// Set a professional color palette
gStyle->SetPalette(kBird); 

// Fix the Z-axis so the mountain doesn't jump around
heat3D->SetMinimum(0);
heat3D->SetMaximum(0.5); 

int nFrames = 400;
for (int i = 1; i <= nFrames; ++i) {
    double t = i * 0.015;
    heat3D->SetParameter(0, t);
    
    // Draw with color-coded surface
    heat3D->Draw("SURF1");
    
    c3->Modified();
    c3->Update();

    // Standard ROOT GIF stitching
    if (i == 1) {
        c3->Print("heat_3d_evolution.gif");
    } else {
        c3->Print("heat_3d_evolution.gif+");
    }
}

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c3
Info in <TCanvas::Print>: gif file heat_3d_evolution.gif has been created


In [8]:
%%html
<img src="heat_3d_evolution.gif" />

In [16]:
// ── Cell 1: Read WAV & plot time domain ──────────────────────────────────────

const char* filename = "chimes.wav";
std::ifstream file(filename, std::ios::binary);
if (!file) { std::cout << "Error: Could not open chimes.wav\n"; return; }

file.seekg(44, std::ios::beg);   // skip standard WAV header

const int N = 65536;             // must be power of 2
std::vector<double> samples;
samples.reserve(N);

short sample_val;
for (int i = 0; i < N && file.read(reinterpret_cast<char*>(&sample_val), sizeof(short)); ++i)
    samples.push_back(static_cast<double>(sample_val));

auto c_audio = new TCanvas("c_audio", "Raw Audio", 800, 400);
TH1D *h_raw = new TH1D("h_raw", "Original Audio (Time Domain);Sample;Amplitude", N, 0, N);
for (int i = 0; i < (int)samples.size(); ++i)
    h_raw->SetBinContent(i+1, samples[i]);
h_raw->Draw();
c_audio->Draw();

In [17]:
// ── Cell 2: Forward FFT & frequency spectrum ─────────────────────────────────
// NOTE: N and samples come from Cell 1 — do NOT re-declare N here

TVirtualFFT *fft = TVirtualFFT::FFT(1, const_cast<int*>(&N), "R2C M K");
fft->SetPoints(samples.data());
fft->Transform();

// Extract complex output into our own arrays so we can modify them safely
std::vector<double> re_arr(N/2 + 1), im_arr(N/2 + 1);
for (int i = 0; i <= N/2; ++i)
    fft->GetPointComplex(i, re_arr[i], im_arr[i]);

auto c_spec = new TCanvas("c_spec", "Frequency Spectrum", 800, 400);
// Label x-axis in Hz assuming 44100 Hz sample rate
double sampleRate = 44100.0;
TH1D *h_spec = new TH1D("h_spec", "Frequency Spectrum;Frequency (Hz);Magnitude",
                         N/2, 0, sampleRate / 2.0);

for (int i = 0; i < N/2; ++i) {
    double mag = std::sqrt(re_arr[i]*re_arr[i] + im_arr[i]*im_arr[i]);
    h_spec->SetBinContent(i+1, mag);
}
h_spec->Draw();
c_spec->Draw();

In [18]:
// ── Cell 3: Filter + Inverse FFT + save WAV ──────────────────────────────────
// Set your frequency cutoff range to DISCARD (in Hz):
double freq_lo =  20000.0;    // <── lower bound of rejected band (Hz)
double freq_hi =  22000.0;  // <── upper bound of rejected band (Hz)
double sampleRate = 44100.0;

In [19]:
// Work on copies of the FFT arrays from Cell 2
std::vector<double> re_filt(re_arr), im_filt(im_arr);

for (int i = 0; i <= N/2; ++i) {
    double freq = i * sampleRate / N;          // bin → Hz
    if (freq >= freq_lo && freq <= freq_hi) {  // zero out the unwanted band
        re_filt[i] = 0.0;
        im_filt[i] = 0.0;
    }
}

// ── Inverse FFT ──────────────────────────────────────────────────────────────
TVirtualFFT *fft_back = TVirtualFFT::FFT(1, const_cast<int*>(&N), "C2R M K");
fft_back->SetPointsComplex(re_filt.data(), im_filt.data());
fft_back->Transform();

// Copy IFFT output into our own array (do NOT pass the internal pointer to Adopt)
std::vector<double> cleaned(N);
for (int i = 0; i < N; ++i)
    cleaned[i] = fft_back->GetPointReal(i) / N;   // FFTW output needs /N normalisation

// ── Plot cleaned waveform ─────────────────────────────────────────────────────
auto c_clean = new TCanvas("c_clean", "Cleaned Audio", 800, 400);
TH1D *h_cleaned = new TH1D("h_cleaned",
    "Cleaned Audio (Time Domain);Sample;Amplitude", N, 0, N);
for (int i = 0; i < N; ++i)
    h_cleaned->SetBinContent(i+1, cleaned[i]);
h_cleaned->Draw();
c_clean->Draw();

// ── Write output WAV ──────────────────────────────────────────────────────────
// Find peak for normalisation (preserve original loudness)
double peak = 0;
for (double v : cleaned) if (std::abs(v) > peak) peak = std::abs(v);
double origPeak = 0;
for (double v : samples) if (std::abs(v) > origPeak) origPeak = std::abs(v);
double scale = (peak > 0) ? origPeak / peak : 1.0;

std::vector<short> out_samples(N);
for (int i = 0; i < N; ++i) {
    double val = cleaned[i] * scale;
    // Clamp to 16-bit range
    val = std::max(-32768.0, std::min(32767.0, val));
    out_samples[i] = static_cast<short>(val);
}

// Build a minimal 44-byte PCM WAV header
const char* outfile = "chimes_processed.wav";
std::ofstream wav(outfile, std::ios::binary);

int   dataSize   = N * sizeof(short);
int   chunkSize  = 36 + dataSize;
short audioFmt   = 1;          // PCM
short numChan    = 1;          // mono
int   byteRate   = (int)sampleRate * 1 * 2;
short blockAlign = 1 * 2;
short bitsPerSmp = 16;

wav.write("RIFF", 4);
wav.write(reinterpret_cast<char*>(&chunkSize),  4);
wav.write("WAVE", 4);
wav.write("fmt ", 4);
int subchunk1 = 16;
wav.write(reinterpret_cast<char*>(&subchunk1),  4);
wav.write(reinterpret_cast<char*>(&audioFmt),   2);
wav.write(reinterpret_cast<char*>(&numChan),    2);
int sr = (int)sampleRate;
wav.write(reinterpret_cast<char*>(&sr),         4);
wav.write(reinterpret_cast<char*>(&byteRate),   4);
wav.write(reinterpret_cast<char*>(&blockAlign), 2);
wav.write(reinterpret_cast<char*>(&bitsPerSmp), 2);
wav.write("data", 4);
wav.write(reinterpret_cast<char*>(&dataSize),   4);
wav.write(reinterpret_cast<char*>(out_samples.data()), dataSize);
wav.close();

std::cout << "Saved: " << outfile
          << "  (" << N << " samples, band " << freq_lo
          << "–" << freq_hi << " Hz removed)\n";

Saved: chimes_processed.wav  (65536 samples, band 20000–22000 Hz removed)


# Leapfrog Integrator

In [4]:
const int n = 1000;
std::vector<double> x(n);
std::vector<double> v(n);
std::vector<double> t(n);
// to be evaluated from the field interp at each step, assuming constant for now
std::vector<double> a(n);
std::vector<double> atm_pad(n);

In [5]:
v.at(0) = 0.0f;
x.at(0) = 0.0f;
t.at(0) = 0.0f;

double dt = 0.1f;

## simulating a rocket flyin off

$$ g = GM / r^2 $$

To simulate ground we can have a no 0 g at surface at 6.3781kms from center of earth, equal to $g(r_{0})$.
With $ r_{o} = 6378.1 km $, we can have rocket acceleration increasing gradually, slowly equalling $g(r_{0})$ for liftoff
and then slowly reaching orbital velocity. the rocket must then tilt slowly in x-direction for the usual path and then see how that goes.

In [6]:
// we are on earth, launching from equator (say, french guyana)

const double G = 6.67e-11; // m3 kg-1 s-2
const double M = 5.9722e+24; // kg
const double R = 6378100.0f; // m
double g = (G * M) / (R * R); // m s-2
double thrust = 0;
x.at(0) = R;

### all these params for the rocket thrust equation
$$\mathbf{F_{thrust}} = \dot{m} v_e + (p_e - p_a) A_e$$

* $\dot{m}$ (mass_flow_rate): The mass of propellant (fuel + oxidizer) expelled per unit time ($\frac{kg}{s}$).
* $v_e$ (exit_velocity): The velocity of the exhaust gas relative to the nozzle exit.
* $\dot{m} v_e$ (Momentum Thrust): The primary thrust component, calculated as mass flow rate multiplied by the exhaust velocity.
* $(p_e - p_a) A_e$ (Pressure Thrust): The additional thrust (or drag) created when the exit pressure ($p_e$: exit_pa) differs from the ambient, or atmospheric, pressure ($p_a$: atm) acting across the nozzle exit area ($A_e$: nozzle_exit_area). [2, 3, 4, 5] 


### we launching ISRO's LVM3

In [7]:
// double m_rocket = 640000; // 640 Tonnes
// double mass_flow_rate = 571.4; // kg s-1 -> constant for a stage
// double exit_velocity = 2550; // m s-1 -> constant for a stage
// double exit_pa = 52000; // pascals -> constant for a stage
// double nozzle_exit_area = 0.8; // m2  -> constant for a stage
// double atm = 101325.0 // pascals  -> decreases exponentially

double mass_flow_rate; // kg s-1 -> constant for a stage
double exit_velocity; // m s-1 -> constant for a stage
double exit_pa; // pascals -> constant for a stage
double nozzle_exit_area; // m2  -> constant for a stage
double thrust_accn = 0;

In [8]:
double m_rocket = 640000; // 640 Tonnes
get_lvm3_stage_properties(0, m_rocket, mass_flow_rate, exit_velocity, exit_pa, nozzle_exit_area);

In [9]:
double atm = get_ambient_pressure(0, R);

In [10]:
atm_pad.at(0) = atm;

In [11]:
// initialization -> half integer velocity
get_lvm3_stage_properties(0, m_rocket, mass_flow_rate, exit_velocity, exit_pa, nozzle_exit_area);
thrust = (mass_flow_rate * exit_velocity)  + (exit_pa - atm) * nozzle_exit_area;
thrust_accn = thrust / m_rocket;
if (thrust_accn > g) {
    a.at(0) = thrust_accn - g;
} else {
    // locked at the launchpad
    a.at(0) = 0.0;
}

double v_half = v.at(0) + a.at(0) * (dt / 2.0f);

for(int i = 0; i < n - 1; i ++) {
    // time evolution leapfrogging (v goes 1/2 integers, and x goes integers, using midpoint velocity)
    x.at(i + 1) = x.at(i) + v_half * dt;
    if (x.at(i + 1) <= R) {
        // rocket cant sink into the earth
        x.at(i + 1) = R;
        v_half = 0.0;
    }
    // as soon as position changes, we need to update the new 'g'
    g = (G * M) / (x.at(i + 1) * x.at(i + 1));
    
    // whats the pressure at this height
    atm = get_ambient_pressure(x.at(i + 1), R);
    // refreshing the thrust related components
    get_lvm3_stage_properties(x.at(i + 1) - R, m_rocket, mass_flow_rate, exit_velocity, exit_pa, nozzle_exit_area);
    // loose some mass (in form of fuel) -> pehle mass loose hoga tabhi accn banega aur calculate hoga
    m_rocket = m_rocket - mass_flow_rate * dt;
    // the standard rocket thrust equation
    thrust = (mass_flow_rate * exit_velocity)  + (exit_pa - atm) * nozzle_exit_area;
    thrust_accn = thrust / m_rocket;
    
    if (thrust_accn <= g && (x.at(i + 1) - R) <= 1e-3) {
        // Rocket does not have enough thrust to lift off yet; remain dead still
        a.at(i + 1) = 0.0;
        // Reset velocity offset to zero
        v_half = 0.0;
        v.at(i + 1) = 0.0;
    } else {
        // Flying or cleared pad: compute acceleration
        a.at(i + 1) = thrust_accn - g;
        
        // Advance velocity from half-step (i + 0.5) to next half-step (i + 1.5)
        v_half = v_half + a.at(i + 1) * dt;
        
        // Sync full-integer step velocity output for tracking/plots
        v.at(i + 1) = v_half - a.at(i + 1) * (dt / 2.0f); 
    }
    
    t.at(i + 1) = t.at(i) + dt;

    // tracking ts for plot later
    atm_pad.at(i + 1) = atm;
}

In [12]:
TGraph* gr = new TGraph(n, x.data(), t.data());
TCanvas* c1 = new TCanvas("c1", "Position vs time");
gr->Draw("ALP");
gr->SetTitle("x vs t in earth accn;t;x");
c1->Draw();

In [13]:
TGraph* gr2 = new TGraph(n, v.data(), t.data());
TCanvas* c2 = new TCanvas("c2", "Velocity vs time");
gr2->Draw("ALP");
gr2->SetTitle("v vs t in earth accn;t;x");
c2->Draw();

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1


In [14]:
TGraph* gr3 = new TGraph(n, atm_pad.data(), t.data());
TCanvas* c3 = new TCanvas("c3", "atm pressure vs time");
gr3->Draw("ALP");
gr3->SetTitle("atm pressure vs t in earth accn;t;x");
c3->Draw();

In [15]:
TGraph* gr4 = new TGraph(n, a.data(), t.data());
TCanvas* c4 = new TCanvas("c4", "a vs time");
gr4->Draw("ALP");
gr4->SetTitle("a vs t in earth accn;t;x");
c4->Draw();